# Adjudication of sepsis patients in EARLI cohort with reasoning models

Here I test GPT-5 on UCSF patient notes

Kernel: `.venv (Python 3.13.5)` via `~/venvs/20240903_LLM_sepsis_prediction`

## Set up packages and API access

In [1]:
import os
from pathlib import Path
import json
import time

import numpy as np
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
from openai import AzureOpenAI

In [2]:
load_dotenv(".env")

API_KEY = os.getenv("API_KEY")
API_VERSION = "2025-04-01-preview"  # latest preview API release
RESOURCE_ENDPOINT = os.getenv("AZURE_ENDPOINT")

client = AzureOpenAI(
    api_key=API_KEY,
    api_version=API_VERSION,
    azure_endpoint=RESOURCE_ENDPOINT,
)
client

Set up paths

In [3]:
project_path = "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Van_research/20240903_LLM_sepsis_prediction"
note_path_ucsf = "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Sepsis_GPT4/Analyses_2025/^Secure_ER_Notes/ER_notes_txt"
note_path_zsfg = "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Sepsis_GPT4/Analyses_2025/^Secure_ER_notes_ZFGH/ER_notes_txt"

## Import metadata

In [4]:
metadata = pl.read_csv(
    "/Users/hoangvanphan/Library/CloudStorage/Box-Box/Sepsis_GPT4/Analyses_2025/EARLI_Metadata/Output/DerivationCohortMetadata012926.csv",
    infer_schema=False,
)

metadata = metadata.select("Barcode","SepsisYN","Group","Hospital_Death")
print(metadata.head())

shape: (5, 4)
┌─────────┬──────────┬───────────────────┬────────────────┐
│ Barcode ┆ SepsisYN ┆ Group             ┆ Hospital_Death │
│ ---     ┆ ---      ┆ ---               ┆ ---            │
│ str     ┆ str      ┆ str               ┆ str            │
╞═════════╪══════════╪═══════════════════╪════════════════╡
│ 10447   ┆ 2        ┆ 3_Sepsis+Cx-      ┆ 0              │
│ 10456   ┆ 1        ┆ 2_Sepsis+OtherCx+ ┆ 0              │
│ 10535   ┆ 3        ┆ 5_Unclear         ┆ 1              │
│ 10584   ┆ 3        ┆ 5_Unclear         ┆ 0              │
│ 10691   ┆ 2        ┆ 3_Sepsis+Cx-      ┆ 0              │
└─────────┴──────────┴───────────────────┴────────────────┘


## Custom functions

API call

In [5]:
def reasoning_adjudication(
        note: str,
        system_prompt: str,
        model: str = "gpt-5-2025-08-07",
        n: int = 1,
        max_completion_tokens: int = 10000,
        reasoning_effort: str = "low",
        # temperature: float = 0.2,  # not used by reasoning models
        # seed: int = 0  # not used by reasoning models
    ):
    # Output: the API response

    messages = [
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": note}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        n=n,
        max_completion_tokens=max_completion_tokens,
        reasoning_effort=reasoning_effort,
    )

    if response.choices[0].finish_reason != "stop":
        raise Exception(f"Finish reason is not 'stop'! It's {response.choices[0].finish_reason}")
    
    return response

## System prompt

In [6]:
sepsis_prompt_20251027 = """
You are an AI clinician with expertise in critical care and/or infectious diseases. You are trying to adjudicate whether a patient is more likely to have critical illness due to any non-infectious cause (No Sepsis), or sepsis (Sepsis) as defined as life-threatening organ dysfunction that is caused by a dysregulated host response to infection. 
Below, you will be given the emergency room note for the patient. I want you to come to your own independent adjudication, regardless of what the clinical team thought. Therefore, you should explicitly ignore whether the patient was receiving antibiotics in your analysis. You should also ignore the clinical team's diagnosis of sepsis, or no sepsis, in your analysis.
End your answer with a paragraph containing either ">>Sepsis" or ">>No Sepsis" to indicate your adjudication, and nothing else.
"""

## GPT-5.2 adjudication

10 adjudications per patient. Save results every 25 patients.

First, exclude the 13 training patients

In [7]:
training_barcodes = [
    "10760",
    "10812",
    "11282",
    "11416",
    "11474",
    "11494",
    "11708",
    "11805",
    "11846",
    "11858",
    "11983",

    "50417",
    "50623",
]

In [8]:
print(metadata.shape)
metadata = metadata.filter(~pl.col("Barcode").is_in(training_barcodes))
print(metadata.shape)

(326, 4)
(313, 4)


### Process all patients in the derivation cohort

Save results every 25 patients

In [9]:
# Iterate
barcodes = metadata["Barcode"].to_list()
df_gpt52 = []

for counter, value in enumerate(barcodes, start=1):
    note_path = note_path_ucsf if value[0] == "1" else note_path_zsfg

    print(f"Processing ID {value}")

    try:
        with open(Path(note_path, f"{value}.txt"), "r", errors="ignore", encoding="utf-8") as f1:
            note = f1.read()
    except FileNotFoundError:
        print(f"***** ID {value} not found *****")
        continue
    
    for k in range(2):
        response = reasoning_adjudication(
            note=note,
            system_prompt=sepsis_prompt_20251027,
            model="gpt-5.2-2025-12-11",
            n=5,
            max_completion_tokens=10000,
            reasoning_effort="high",
        )

        for j in range(len(response.choices)):
            temp = {"Barcode": value, "run": k+1}
            temp.update(
                {"adjudication": response.choices[j].message.content,
                 "sepsis": response.choices[j].message.content.split(">>")[-1].strip()}
            )
            df_gpt52.append(temp)

        time.sleep(3)  # add sleep so that it doesn't overwhelm the requests per minute limit

    if (counter % 25) == 0:
        pl.DataFrame(df_gpt52).write_csv(
            Path(project_path, "Secure_out", f"20260319_GPT5.2_sepsis_group{counter//25}.csv")
        )
        df_gpt52 = []

Processing ID 10447
Processing ID 10456
Processing ID 10535
Processing ID 10584
Processing ID 10691
Processing ID 10740
Processing ID 10741
Processing ID 10755
Processing ID 10757
Processing ID 10778
Processing ID 10819
Processing ID 10820
Processing ID 10825
Processing ID 10846
Processing ID 10856
Processing ID 10902
Processing ID 10929
Processing ID 10956
Processing ID 10957
Processing ID 10958
Processing ID 10963
Processing ID 10971
Processing ID 10979
Processing ID 10981
Processing ID 10983
Processing ID 10991
Processing ID 10993
Processing ID 11005
Processing ID 11010
Processing ID 11011
Processing ID 11017
Processing ID 11022
Processing ID 11024
Processing ID 11028
Processing ID 11035
Processing ID 11038
Processing ID 11043
Processing ID 11046
Processing ID 11052
Processing ID 11053
Processing ID 11062
Processing ID 11079
Processing ID 11080
Processing ID 11086
Processing ID 11093
Processing ID 11097
Processing ID 11111
Processing ID 11114
Processing ID 11122
Processing ID 11128


Process encrypted notes

In [10]:
# Iterate
# barcodes = metadata["Barcode"].to_list()
df_gpt52 = []

for value in ["11309"]:

    print(f"Processing ID {value}")

    try:
        with open(Path("/Users/hoangvanphan/Library/Containers/com.ciphercloud.macapp.CipherCloud/Data/Library/Caches/com.ciphercloud.macapp.CipherCloud/originalFile", f"{value}.txt"), "r", errors="ignore", encoding="utf-8") as f1:
            note = f1.read()
    except FileNotFoundError:
        print(f"***** ID {value} not found *****")
        continue
    
    for k in range(2):
        response = reasoning_adjudication(
            note=note,
            system_prompt=sepsis_prompt_20251027,
            model="gpt-5.2-2025-12-11",
            n=5,
            max_completion_tokens=10000,
            reasoning_effort="high",
        )

        for j in range(len(response.choices)):
            temp = {"Barcode": value, "run": k+1}
            temp.update(
                {"adjudication": response.choices[j].message.content,
                 "sepsis": response.choices[j].message.content.split(">>")[-1].strip()}
            )
            df_gpt52.append(temp)

        time.sleep(3)  # add sleep so that it doesn't overwhelm the requests per minute limit

Processing ID 11309


In [11]:
# Iterate
# barcodes = metadata["Barcode"].to_list()
# df_gpt52 = []

for value in ["50372"]:

    print(f"Processing ID {value}")

    try:
        with open(Path("/Users/hoangvanphan/Library/Containers/com.ciphercloud.macapp.CipherCloud/Data/Library/Caches/com.ciphercloud.macapp.CipherCloud/originalFile", f"{value}.txt"), "r", errors="ignore", encoding="utf-8") as f1:
            note = f1.read()
    except FileNotFoundError:
        print(f"***** ID {value} not found *****")
        continue
    
    for k in range(2):
        response = reasoning_adjudication(
            note=note,
            system_prompt=sepsis_prompt_20251027,
            model="gpt-5.2-2025-12-11",
            n=5,
            max_completion_tokens=10000,
            reasoning_effort="high",
        )

        for j in range(len(response.choices)):
            temp = {"Barcode": value, "run": k+1}
            temp.update(
                {"adjudication": response.choices[j].message.content,
                 "sepsis": response.choices[j].message.content.split(">>")[-1].strip()}
            )
            df_gpt52.append(temp)

        time.sleep(3)  # add sleep so that it doesn't overwhelm the requests per minute limit

Processing ID 50372


Export the 2 patients

In [12]:
pl.DataFrame(df_gpt52).write_csv(
    Path(project_path, "Secure_out", "20260319_GPT5.2_sepsis_group13.csv")
)

I forgot to save the answer for the last group of 25 patients!

In [13]:
# Iterate
barcodes = metadata["Barcode"].to_list()
df_gpt52 = []

for counter, value in enumerate(barcodes[300:], start=301):
    note_path = note_path_ucsf if value[0] == "1" else note_path_zsfg

    print(f"Processing ID {value}")

    try:
        with open(Path(note_path, f"{value}.txt"), "r", errors="ignore", encoding="utf-8") as f1:
            note = f1.read()
    except FileNotFoundError:
        print(f"***** ID {value} not found *****")
        continue
    
    for k in range(2):
        response = reasoning_adjudication(
            note=note,
            system_prompt=sepsis_prompt_20251027,
            model="gpt-5.2-2025-12-11",
            n=5,
            max_completion_tokens=10000,
            reasoning_effort="high",
        )

        for j in range(len(response.choices)):
            temp = {"Barcode": value, "run": k+1}
            temp.update(
                {"adjudication": response.choices[j].message.content,
                 "sepsis": response.choices[j].message.content.split(">>")[-1].strip()}
            )
            df_gpt52.append(temp)

        time.sleep(3)  # add sleep so that it doesn't overwhelm the requests per minute limit


pl.DataFrame(df_gpt52).write_csv(
    Path(project_path, "Secure_out", "20260319_GPT5.2_sepsis_group14.csv")
)

Processing ID 50643
Processing ID 50647
Processing ID 50650
Processing ID 50660
Processing ID 50661
Processing ID 50680
Processing ID 50681
Processing ID 50685
***** ID 50685 not found *****
Processing ID 50692
Processing ID 50722
Processing ID 50723
Processing ID 50725
Processing ID 50749
